# Session 13 — 20M LLM / 50M Tokens / Reversible Training

This notebook runs the three assignment experiments:

1. baseline Transformer at a fixed batch;
2. reversible Transformer at the same batch;
3. reversible Transformer at the maximum batch that fits.

The reversible implementation uses exact additive coupling and reconstructs the reversible stack during backward instead of storing every depth activation.

In [ ]:
!pip -q install datasets tiktoken pandas matplotlib tabulate
!rm -rf /content/ERA-V5_Hari
!git clone -q https://github.com/bedathurihari/ERA-V5_Hari.git /content/ERA-V5_Hari
%cd /content/ERA-V5_Hari/Session_13/Assignment_13

import torch, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from session13_core import *
from session13_validation import reversibility_test

cfg = Config()
seed_all()
device = get_device()
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
print(cfg)

## Prepare the released corpus

TinyStories is streamed and tokenized with the GPT-2 BPE. The token files are cached under `/content/session13_data`.

In [ ]:
train_bin, val_bin, meta = prepare_data(cfg)
print(meta)

## Verify model size and reversibility

The baseline and reversible models must have the same trainable parameter count. The reversibility test checks reconstruction and gradient agreement before the expensive runs.

In [ ]:
baseline = BaselineLM(cfg)
reversible = ReversibleLM(cfg)
print("baseline params:", f"{count_params(baseline):,}", f"({count_params(baseline)/1e6:.2f}M)")
print("reversible params:", f"{count_params(reversible):,}", f"({count_params(reversible)/1e6:.2f}M)")
assert count_params(baseline) == count_params(reversible)
del baseline, reversible

print(reversibility_test(device))

## Find the maximum reversible batch

The probe uses real forward + backward passes. It doubles the batch until OOM and then refines the boundary.

In [ ]:
max_rev_batch, batch_probe = find_max_batch(
    ReversibleLM, cfg, train_bin, start=cfg.fixed_batch_size
)
display(batch_probe)
print("maximum reversible batch:", max_rev_batch)

## Run A — baseline, fixed batch

Each run consumes at least 50,000,000 tokens and writes checkpoints plus CSV/JSON evidence.

In [ ]:
baseline_summary = train_experiment(
    "baseline_fixed_batch",
    BaselineLM,
    cfg.fixed_batch_size,
    cfg,
    train_bin,
    val_bin,
)
baseline_summary

## Run B — reversible, same fixed batch

In [ ]:
reversible_fixed_summary = train_experiment(
    "reversible_fixed_batch",
    ReversibleLM,
    cfg.fixed_batch_size,
    cfg,
    train_bin,
    val_bin,
)
reversible_fixed_summary

## Run C — reversible, maximum batch

In [ ]:
reversible_max_summary = train_experiment(
    "reversible_max_batch",
    ReversibleLM,
    max_rev_batch,
    cfg,
    train_bin,
    val_bin,
)
reversible_max_summary

## Final evidence table and plots

These are the values to copy into the README after the complete GPU run. Do not replace them with estimates.

In [ ]:
results = pd.DataFrame([
    baseline_summary,
    reversible_fixed_summary,
    reversible_max_summary,
])
display(results)
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
results.to_csv(Path(cfg.output_dir) / "summary.csv", index=False)

cols = [
    "name", "model_params", "batch_size", "tokens_seen",
    "final_train_loss", "final_val_loss",
    "avg_tokens_per_second", "peak_memory_GiB",
    "wall_clock_minutes", "gpu"
]
print(results[cols].to_markdown(index=False))

for metric, ylabel in [
    ("peak_memory_GiB", "CUDA peak allocated memory (GiB)"),
    ("avg_tokens_per_second", "Average tokens / second"),
]:
    plt.figure(figsize=(8,5))
    results.set_index("name")[metric].plot(kind="bar")
    plt.ylabel(ylabel)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(9,5))
for name in ["baseline_fixed_batch","reversible_fixed_batch","reversible_max_batch"]:
    df = pd.read_csv(Path(cfg.output_dir) / name / "train_log.csv")
    plt.plot(df["tokens_seen"]/1e6, df["train_loss"], label=name)
plt.xlabel("Tokens seen (millions)")
plt.ylabel("Training loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Interpretation

I report the fixed-batch comparison first because it isolates the memory/speed cost of reversibility. The max-batch run then answers whether the saved activation memory is actually useful.

The reversible method is a compute-for-memory trade: backward recomputes attention and MLP functions, so lower memory is expected to cost throughput. The result is only meaningful when I report both numbers together.